# 04 — Optimizers — Hands-on Tutorial

In this notebook you will:
- Visualise how SGD, Momentum, and Adam navigate a 2D loss surface
- Trace Adam's per-parameter adaptive step sizes through training
- Compare common learning rate schedules: StepLR, CosineAnnealingLR, ReduceLROnPlateau
- Implement early stopping with a patience parameter

**Prior:** Notebooks 01–03 complete; UL notebook 03 (gradient descent on 2D surfaces)

## Where this fits

| # | Topic | Slide deck | Notebook |
|---|---|---|---|
| 1 | Single neuron | `01_single_neuron.pdf` | `01_single_neuron.ipynb` |
| 2 | Multilayer networks | `02_multilayer_networks.pdf` | `02_training_loop.ipynb` |
| 3 | Backpropagation | `03_backprop_training.pdf` | `03_backpropagation.ipynb` |
| **→ 4** | **Optimizers** | **`07_optimizers.pdf` *(new)*** | **`04_optimizers.ipynb`** |
| 5 | CNNs | `04_cnns.pdf` | `05_cnns.ipynb` |
| 6 | Modern architectures | `05a_attention.pdf` + `05b_practical.pdf` | *(bonus: `bonus_generative_models.ipynb`)* |
| 7 | Bayesian inference | `06_bayesian_inference.pdf` | `06_bayesian_inference.ipynb` |

**Coming from:** Notebook 03 explained where gradients come from; now you'll see how different optimizers turn those gradients into parameter updates.

**Leading to:** Notebooks 05 and 06 lean on the optimizer choices and learning-rate intuition you build here.

**If you skipped ahead:** The four-step training loop from notebook 02 is the only hard prerequisite.


In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────
# Run this cell only on Colab. On JupyterHub, packages are pre-installed.
import sys
if 'google.colab' in sys.modules:
    %pip install -q ipywidgets torch

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.optim as optim
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

%matplotlib inline

---

## The mathematics of optimizers

All three optimizers below update parameters using gradients — they differ in *how* they use gradient history.

### SGD (vanilla)

$$w_t \leftarrow w_t - \eta \, g_t$$

### SGD with Momentum

$$v_t = \beta\, v_{t-1} + g_t, \qquad w_t \leftarrow w_t - \eta\, v_t$$

### Adam

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\,g_t \qquad \text{(1st moment — mean)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2 \qquad \text{(2nd moment — variance)}$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t} \qquad \text{(bias correction)}$$
$$w_t \leftarrow w_t - \eta\, \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

| Symbol | Meaning |
|--------|---------|
| $g_t = \nabla_w L$ | Gradient at step $t$ |
| $\eta$ | Learning rate |
| $\beta \approx 0.9$ | Momentum coefficient — how much to remember previous direction |
| $\beta_1 \approx 0.9$ | Decay rate for gradient mean |
| $\beta_2 \approx 0.999$ | Decay rate for gradient variance |
| $\epsilon \approx 10^{-8}$ | Small constant to prevent division by zero |
| $\hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$ | Normalised gradient — step size adapts per parameter |

Adam's key property: a parameter receiving large, consistent gradients gets a *smaller* effective step; one receiving small, noisy gradients gets a *larger* step.

In [ ]:
# ── Implement all three optimizers from scratch; plot on the same loss ────────

# Loss: f(w1, w2) = w1^2 + 10*w2^2  (elongated bowl, same as Part 1 widget)
def grad_f(w): return np.array([2*w[0], 20*w[1]])

def run_sgd_scratch(start, lr, n):
    w, traj, losses = np.array(start, float), [start], []
    for _ in range(n):
        g = grad_f(w);  w = w - lr * g
        traj.append(w.copy()); losses.append(w[0]**2 + 10*w[1]**2)
    return np.array(traj), losses

def run_momentum_scratch(start, lr, beta, n):
    w, v = np.array(start, float), np.zeros(2)
    traj, losses = [start], []
    for _ in range(n):
        g = grad_f(w);  v = beta*v + g;  w = w - lr*v
        traj.append(w.copy()); losses.append(w[0]**2 + 10*w[1]**2)
    return np.array(traj), losses

def run_adam_scratch(start, lr, b1, b2, eps, n):
    w, m, v = np.array(start, float), np.zeros(2), np.zeros(2)
    traj, losses = [start], []
    for t in range(1, n+1):
        g = grad_f(w)
        m = b1*m + (1-b1)*g
        v = b2*v + (1-b2)*g**2
        m_hat = m / (1 - b1**t)
        v_hat = v / (1 - b2**t)
        w = w - lr * m_hat / (np.sqrt(v_hat) + eps)
        traj.append(w.copy()); losses.append(w[0]**2 + 10*w[1]**2)
    return np.array(traj), losses

start = [-3.0, 3.0]
n     = 80

traj_sgd,  loss_sgd  = run_sgd_scratch(start, lr=0.05, n=n)
traj_mom,  loss_mom  = run_momentum_scratch(start, lr=0.05, beta=0.9, n=n)
traj_adam, loss_adam = run_adam_scratch(start, lr=0.15, b1=0.9, b2=0.999, eps=1e-8, n=n)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
steps = np.arange(1, n+1)
ax1.semilogy(steps, loss_sgd,  lw=2, label='SGD')
ax1.semilogy(steps, loss_mom,  lw=2, label='Momentum (β=0.9)')
ax1.semilogy(steps, loss_adam, lw=2, label='Adam')
ax1.set_xlabel('Step'); ax1.set_ylabel('Loss  (log scale)')
ax1.set_title('Convergence from scratch implementations')
ax1.legend(); ax1.grid(alpha=0.3)

# Effective step size — what Adam actually divides by
g_const   = np.ones(100) * 2.0          # constant gradient
g_varying = np.concatenate([10*np.ones(50), 0.2*np.ones(50)])  # large then small

def adam_eff_step(grads, b1=0.9, b2=0.999, eps=1e-8):
    m, v, eff = 0, 0, []
    for t, g in enumerate(grads, 1):
        m = b1*m + (1-b1)*g
        v = b2*v + (1-b2)*g**2
        m_hat = m / (1 - b1**t)
        v_hat = v / (1 - b2**t)
        eff.append(abs(m_hat) / (np.sqrt(v_hat) + eps))
    return np.array(eff)

ax2.plot(adam_eff_step(g_const),   lw=2, label='Constant gradient = 2')
ax2.plot(adam_eff_step(g_varying), lw=2, label='Large (10) then small (0.2) gradient')
ax2.axvline(50, color='gray', ls='--', lw=1, label='Gradient switches at step 50')
ax2.set_xlabel('Step'); ax2.set_ylabel('Effective step  $|\hat{m}_t| / (\sqrt{\hat{v}_t}+\epsilon)$')
ax2.set_title("Adam's adaptive normalisation\nlarge gradients → smaller effective step")
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()


### Think about it

- In the momentum update $v_t = \beta v_{t-1} + g_t$, what does $v_t$ converge   to if the gradient is constant? Does this mean momentum overshoots the minimum?
- Adam's effective step is $\hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$.   If a parameter has a large, consistent gradient, is its effective step larger   or smaller than vanilla SGD with the same $\eta$? What about a noisy, small gradient?
- The bias correction terms $(1 - \beta_1^t)$ and $(1 - \beta_2^t)$ are close to   zero at $t=1$ and approach 1 as $t \to \infty$. What would happen at early steps   without this correction?
- The Adam right-hand plot shows the effective step dropping when a large gradient   arrives. In physical terms: if a galaxy survey has a sudden influx of unusual   objects (large gradients), what is Adam doing to the learning rate for that feature?

---

## Part 1 — SGD, Momentum, and Adam: navigating a loss surface

You have seen gradient descent find a minimum on a 2D surface (UL notebook 03).
Now we compare three optimisers on the same surface simultaneously.

**Loss surface:** $f(w_1, w_2) = w_1^2 + 10\,w_2^2$ — an elongated bowl where the curvature
in the $w_2$ direction is 10× steeper than in $w_1$. This is a deliberately simple model of
what happens in practice: different parameters have very different gradient magnitudes.

**Starting point:** $(-3, 3)$. **Minimum:** $(0, 0)$.

All three optimisers start at the same point and use the same learning rate.
Watch what the geometry of the surface does to each one.

In [ ]:
# ── Part 1: loss surface helpers and trajectory functions ───────────────────

def f(x, y):
    """Elongated bowl: curvature 10× steeper in y than x."""
    return x**2 + 10 * y**2

def grad_f(x, y):
    return np.array([2 * x, 20 * y])

def run_sgd(start, lr, n):
    w = np.array(start, dtype=float)
    traj = [w.copy()]
    for _ in range(n):
        w = w - lr * grad_f(*w)
        traj.append(w.copy())
    return np.array(traj)

def run_momentum(start, lr, n, beta=0.9):
    w = np.array(start, dtype=float)
    v = np.zeros(2)
    traj = [w.copy()]
    for _ in range(n):
        g = grad_f(*w)
        v = beta * v + g
        w = w - lr * v
        traj.append(w.copy())
    return np.array(traj)

def run_adam(start, lr, n, beta1=0.9, beta2=0.999, eps=1e-8):
    w = np.array(start, dtype=float)
    m, v = np.zeros(2), np.zeros(2)
    traj = [w.copy()]
    for t in range(1, n + 1):
        g = grad_f(*w)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        w = w - lr * m_hat / (np.sqrt(v_hat) + eps)
        traj.append(w.copy())
    return np.array(traj)

def plot_trajectories(lr=0.05, n_steps=60):
    start = [-3.0, 3.0]
    sgd  = run_sgd(start, lr, n_steps)
    mom  = run_momentum(start, lr, n_steps)
    adam = run_adam(start, lr, n_steps)

    x_grid = np.linspace(-4, 0.5, 300)
    y_grid = np.linspace(-0.5, 4, 300)
    X, Y = np.meshgrid(x_grid, y_grid)
    Z = f(X, Y)

    fig, ax = plt.subplots(figsize=(8, 6))
    cs = ax.contourf(X, Y, Z, levels=25, cmap='Blues_r', alpha=0.8)
    ax.contour(X, Y, Z, levels=25, colors='white', alpha=0.25, linewidths=0.5)

    ax.plot(sgd[:, 0],  sgd[:, 1],  'r-o', ms=3, lw=1.5, label='SGD',              alpha=0.9)
    ax.plot(mom[:, 0],  mom[:, 1],  'g-o', ms=3, lw=1.5, label='Momentum (\u03b2=0.9)', alpha=0.9)
    ax.plot(adam[:, 0], adam[:, 1], 'y-o', ms=3, lw=1.5, label='Adam',             alpha=0.9)
    ax.plot(*start, 'k*', ms=12, label='Start')
    ax.plot(0, 0, 'w*', ms=12, label='Minimum')

    ax.set_xlabel('w\u2081')
    ax.set_ylabel('w\u2082')
    ax.set_title(f'Loss surface: f(w\u2081,w\u2082) = w\u2081\u00b2 + 10w\u2082\u00b2  |  lr={lr:.3f}, {n_steps} steps')
    ax.legend(fontsize=9)
    plt.colorbar(cs, ax=ax, label='Loss')
    plt.tight_layout()
    plt.show()

interact(
    plot_trajectories,
    lr=FloatSlider(value=0.05, min=0.001, max=0.15, step=0.001,
                   description='lr', readout_format='.3f'),
    n_steps=IntSlider(value=60, min=10, max=150, step=10, description='steps'),
);

### Think about it

- Set `lr=0.15`. What happens to SGD? Does Momentum also diverge, or does it recover?
  What property of Momentum prevents (or delays) divergence?
- On this elongated surface, which optimiser reaches the minimum in the fewest steps?
  Would the answer change if you made the surface rounder (reducing the 10 factor toward 1)?
- Momentum averages the last ~10 gradient directions (at $\beta=0.9$). What does that mean
  for a loss surface with sharp curves — does averaging help or hurt?
- A galaxy morphology classifier has some layers with large activations and some with near-zero.
  Which optimiser is better placed to handle parameters with very different gradient scales?

---

### Convergence curves: loss vs step

The trajectory widget shows *where* in parameter space each optimizer goes. This plot shows *how fast* each one reduces the loss — the same run, three optimizers, on the same axes.

Two views: **loss vs epoch** (wall-clock proxy) and **loss vs gradient evaluation** (compute-cost proxy). They can tell different stories.

In [ ]:
# ── Convergence curves: SGD, Momentum, Adam on the same axes ─────────────────

def run_optimizer_curve(opt_name, start=(-3.0, 3.0), lr=0.05, n_steps=200):
    """Run optimizer on f(w) = w1^2 + 10*w2^2, return loss per step."""
    w = torch.tensor(start, dtype=torch.float32, requires_grad=True)
    
    if opt_name == 'SGD':
        opt = torch.optim.SGD([w], lr=lr)
    elif opt_name == 'Momentum':
        opt = torch.optim.SGD([w], lr=lr, momentum=0.9)
    elif opt_name == 'Adam':
        opt = torch.optim.Adam([w], lr=lr)
    
    losses = []
    for _ in range(n_steps):
        opt.zero_grad()
        loss = w[0]**2 + 10 * w[1]**2
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

lr_common = 0.05
curves = {
    'SGD':      run_optimizer_curve('SGD',      lr=lr_common),
    'Momentum': run_optimizer_curve('Momentum', lr=lr_common),
    'Adam':     run_optimizer_curve('Adam',     lr=lr_common),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
colors = {'SGD': 'steelblue', 'Momentum': 'darkorange', 'Adam': 'green'}

for name, losses in curves.items():
    ax1.plot(losses, label=name, color=colors[name], lw=2)
    ax2.semilogy(losses, label=name, color=colors[name], lw=2)

for ax in (ax1, ax2):
    ax.set_xlabel('Step')
    ax.legend()
    ax.axhline(0, color='gray', lw=0.5, ls='--')

ax1.set_ylabel('Loss  f(w)')
ax1.set_title(f'Convergence — linear scale  (lr={lr_common})')
ax2.set_ylabel('Loss  f(w)  [log scale]')
ax2.set_title('Same curves — log scale reveals late-stage behaviour')

plt.tight_layout()
plt.show()

print('Steps to reach loss < 0.01:')
for name, losses in curves.items():
    steps = next((i for i, l in enumerate(losses) if l < 0.01), None)
    print(f'  {name:12s}: {steps if steps else ">200"} steps')


### Think about it

- On the log-scale plot, which optimizer reaches $10^{-4}$ first? Does the   ordering change at different loss thresholds?
- Momentum converges faster than SGD on this elongated bowl. Does it overshoot   at the beginning? Can you see the oscillation on the linear plot?
- Adam uses a fixed learning rate of 0.05 here. Try changing `lr_common` to 0.2   in the cell above. Which optimizer is most sensitive to this change?
- On a real problem (not a clean quadratic bowl), the log-scale convergence   curve often shows a **knee** — rapid early descent then slow tail.   Which optimizer tends to have the flattest tail?

---

## Part 2 — Adam's adaptive step sizes

One of Adam's key properties is that it normalises the effective step for each parameter
independently. At step $t$, Adam updates:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1)\,g_t \qquad
  v_t = \beta_2 v_{t-1} + (1 - \beta_2)\,g_t^2$$

$$\Delta w = -\,\frac{\alpha \cdot m_t / (1 - \beta_1^t)}{\sqrt{v_t / (1 - \beta_2^t)}\; +\; \epsilon}$$

The denominator tracks the *root-mean-square gradient* for each parameter.
A parameter that consistently receives large gradients accumulates a large $v_t$,
which shrinks its effective step — preventing it from dominating training.

Below, two parameters receive gradients of very different magnitudes.
Compare how vanilla SGD and Adam handle them.

### Adam by hand — watch the bias correction with real numbers

Before the interactive plot, let us run Adam on **one parameter** for ten steps
and *print every intermediate number*. This is the whole algorithm — no library.

We feed a **constant gradient** $g = 1.0$ at every step. That is deliberate: if the
gradient is always exactly $1.0$, then the true mean of $g$ is $1.0$ and the true
mean of $g^2$ is $1.0$. So we know the *right* answers in advance — the running
averages $m_t, v_t$ should both head toward $1.0$.

The catch: Adam initialises $m_0 = 0$ and $v_0 = 0$. Early on the running averages
are dragged toward that zero start — they are **biased low**. The correction factors
$1-\beta_1^{\,t}$ and $1-\beta_2^{\,t}$ exist to undo exactly this.

**Step 1, by hand** (with $\beta_1=0.9,\ \beta_2=0.999$):

$$m_1 = 0.9\cdot 0 + 0.1\cdot 1.0 = 0.1 \quad\text{(raw — 10x too small!)}$$
$$\hat m_1 = \frac{m_1}{1-\beta_1^{1}} = \frac{0.1}{0.1} = 1.0 \quad\text{(corrected — exactly right)}$$

$$v_1 = 0.999\cdot 0 + 0.001\cdot 1.0^2 = 0.001,\qquad
   \hat v_1 = \frac{0.001}{1-\beta_2^{1}} = \frac{0.001}{0.001} = 1.0$$

So the corrected effective step at $t=1$ is $\alpha\cdot \hat m_1/\sqrt{\hat v_1}
= \alpha\cdot 1.0/1.0 = \alpha$ — the intended step. Watch what the *uncorrected*
version would have done instead.

In [ ]:
# -- Adam by hand: print m_t, v_t, the corrections, and the effective step ----
beta1, beta2, eps, alpha = 0.9, 0.999, 1e-8, 0.001
g = 1.0                 # constant gradient -> true mean(g)=1.0, true mean(g^2)=1.0
m = v = 0.0             # both start at zero: this is the source of the bias

hdr = (f"{'t':>2} | {'g':>4} | {'m_t (raw)':>10} {'1-b1^t':>7} {'m_hat':>7} | "
       f"{'v_t (raw)':>10} {'1-b2^t':>7} {'v_hat':>7} | "
       f"{'step (raw)':>10} {'step (corr)':>11}")
print(hdr); print('-' * len(hdr))

for t in range(1, 11):
    m = beta1 * m + (1 - beta1) * g          # first moment  (running mean of g)
    v = beta2 * v + (1 - beta2) * g**2       # second moment (running mean of g^2)
    c1, c2 = 1 - beta1**t, 1 - beta2**t      # bias-correction denominators
    m_hat, v_hat = m / c1, v / c2            # corrected estimates

    step_raw  = alpha * m     / (v**0.5     + eps)   # WITHOUT bias correction
    step_corr = alpha * m_hat / (v_hat**0.5 + eps)   # WITH bias correction

    print(f"{t:>2} | {g:>4.1f} | {m:>10.5f} {c1:>7.4f} {m_hat:>7.4f} | "
          f"{v:>10.6f} {c2:>7.4f} {v_hat:>7.4f} | "
          f"{step_raw:>10.6f} {step_corr:>11.6f}")

print(f"\nlr (alpha) = {alpha}.  With correction the step is ~alpha from t=1 on.")
print("Without correction the first step is ~3x too large, then wanders in.")

**Read the two rightmost columns.**

- **step (corr)** — the real Adam step — sits at ~ `alpha` (0.001) from the very
  first step. The bias correction makes Adam behave correctly *immediately*.
- **step (raw)** — Adam *without* bias correction — starts around `0.0032`
  (~3x too large, because $\sqrt{v_1}=\sqrt{0.001}\approx 0.0316$ while $m_1=0.1$, so $m_1/\sqrt{v_1}\approx 3.16$),
  and only drifts toward the right value after many steps.

That factor of $1-\beta_2^{\,t}$ matters most because $\beta_2=0.999$ is so close to
1: $v_t$ crawls up from zero, so $\sqrt{v_t}$ is tiny early and the *uncorrected*
step explodes. Bias correction is not a cosmetic detail — it is what stops Adam
taking a wild first step before it has seen enough gradients.

**Try it:** change `g = 1.0` to a constant `5.0`. The raw columns change, but
**step (corr)** still settles at ~ `alpha` — Adam normalises away the gradient
*scale*, keeping the step governed by `lr` alone.

In [ ]:
# ── Part 2: Adam's per-parameter adaptive step sizes ────────────────────────

def plot_adam_adaptation(beta1=0.90, beta2=0.999, n_steps=100):
    """
    Param 1: constant gradient of 1.0.
    Param 2: gradient of 10.0 for the first half, then 0.1 for the second half.
    Shows how Adam normalises effective step sizes; contrast with vanilla SGD.
    """
    lr, eps = 0.01, 1e-8
    split = n_steps // 2

    g1 = np.ones(n_steps)
    g2 = np.concatenate([10 * np.ones(split), 0.1 * np.ones(n_steps - split)])

    m1, v1, m2, v2 = 0.0, 0.0, 0.0, 0.0
    eff1, eff2, raw1, raw2 = [], [], [], []

    for t in range(1, n_steps + 1):
        g1t, g2t = g1[t - 1], g2[t - 1]

        m1 = beta1 * m1 + (1 - beta1) * g1t
        v1 = beta2 * v1 + (1 - beta2) * g1t**2
        m2 = beta1 * m2 + (1 - beta1) * g2t
        v2 = beta2 * v2 + (1 - beta2) * g2t**2

        m1h = m1 / (1 - beta1**t);  v1h = v1 / (1 - beta2**t)
        m2h = m2 / (1 - beta1**t);  v2h = v2 / (1 - beta2**t)

        eff1.append(abs(lr * m1h / (np.sqrt(v1h) + eps)))
        eff2.append(abs(lr * m2h / (np.sqrt(v2h) + eps)))
        raw1.append(lr * abs(g1t))
        raw2.append(lr * abs(g2t))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(raw1, 'r-', lw=2, label='Param 1  (g = 1.0)')
    axes[0].plot(raw2, 'b-', lw=2, label='Param 2  (g = 10 \u2192 0.1)')
    axes[0].axvline(split, ls='--', c='gray', alpha=0.6, label='Gradient change')
    axes[0].set_xlabel('Step');  axes[0].set_ylabel('|\u0394w| = lr \u00b7 |g|')
    axes[0].set_title('Vanilla SGD: step size \u221d gradient magnitude')
    axes[0].legend()

    axes[1].plot(eff1, 'r-', lw=2, label='Param 1  (g = 1.0)')
    axes[1].plot(eff2, 'b-', lw=2, label='Param 2  (g = 10 \u2192 0.1)')
    axes[1].axvline(split, ls='--', c='gray', alpha=0.6, label='Gradient change')
    axes[1].set_xlabel('Step');  axes[1].set_ylabel('Effective |\u0394w|')
    axes[1].set_title(f'Adam (\u03b2\u2081={beta1:.2f}, \u03b2\u2082={beta2:.3f}): step sizes equalise over time')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

interact(
    plot_adam_adaptation,
    beta1=FloatSlider(value=0.90, min=0.5, max=0.99, step=0.01,
                      description='\u03b2\u2081', readout_format='.2f'),
    beta2=FloatSlider(value=0.999, min=0.9, max=0.9999, step=0.001,
                      description='\u03b2\u2082', readout_format='.3f'),
    n_steps=IntSlider(value=100, min=20, max=200, step=10, description='steps'),
);

### Think about it

- At step $t=1$, the bias-corrected estimates are $m_1 / (1 - \beta_1)$ and
  $v_1 / (1 - \beta_2)$. With $\beta_1=0.9$ and $\beta_2=0.999$, what is Adam's
  effective step at step 1 for a constant gradient of 1.0? Is it larger or smaller than `lr`?
- Set $\beta_2 = 0.9$ (much lower). The second moment accumulates faster — what happens
  to the effective step size? Does Adam still normalise param 2's large gradients?
- A parameter in a galaxy classifier always receives near-zero gradients throughout training.
  How does Adam handle this differently from SGD?
- After the gradient change at the halfway point, Param 2's effective step takes many steps
  to adapt. What controls the adaptation speed — $\beta_1$, $\beta_2$, or both?

---

## Part 3 — Learning rate schedules

A fixed learning rate is rarely optimal:
large early on for fast progress, small later for fine-grained convergence.
Learning rate schedules automate this.

Three common schedules:

| Schedule | Rule | When to use |
|---|---|---|
| **StepLR** | Multiply lr by $\gamma$ every `step_size` epochs | Predictable training curves |
| **CosineAnnealingLR** | Decay lr along a cosine curve over `T_max` epochs | Common default; smooth |
| **ReduceLROnPlateau** | Reduce lr by $\gamma$ if val loss plateaus for `patience` epochs | When you don't know the epoch count |

The `*` labels on the sliders below indicate which hyperparameters apply to each schedule:
`step_size*` and `T_max*` are schedule-specific; $\gamma$ and `patience*` are used by different schedules.

In [ ]:
# ── Part 3: Learning rate schedules ─────────────────────────────────────────

def plot_lr_schedule(schedule='cosine', initial_lr=0.1, n_epochs=100,
                     step_size=20, gamma=0.5, T_max=50, patience=10):
    dummy = torch.tensor([0.0], requires_grad=True)
    opt = optim.SGD([dummy], lr=initial_lr)

    names = {'step': 'StepLR', 'cosine': 'CosineAnnealingLR', 'plateau': 'ReduceLROnPlateau'}

    if schedule == 'step':
        sched = optim.lr_scheduler.StepLR(opt, step_size=int(step_size), gamma=gamma)
    elif schedule == 'cosine':
        sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=int(T_max))
    else:
        sched = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=int(patience), factor=gamma)

    np.random.seed(0)
    val_loss = (np.exp(-np.linspace(0, 3, n_epochs))
                + 0.04 * np.random.randn(n_epochs))
    val_loss = np.clip(val_loss, 0.02, None)

    lrs = [initial_lr]
    for epoch in range(n_epochs):
        if schedule == 'plateau':
            sched.step(float(val_loss[epoch]))
        else:
            sched.step()
        lrs.append(opt.param_groups[0]['lr'])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(lrs, 'b-', lw=2)
    axes[0].set_xlabel('Epoch');  axes[0].set_ylabel('Learning rate')
    axes[0].set_title(names[schedule])
    if min(lrs) > 0:
        axes[0].set_yscale('log')

    axes[1].plot(val_loss, 'r-', lw=2, label='val loss (simulated)')
    ax2 = axes[1].twinx()
    ax2.plot(lrs[:n_epochs], 'b--', alpha=0.7, lw=2, label='lr')
    axes[1].set_xlabel('Epoch');  axes[1].set_ylabel('Val loss', color='r')
    ax2.set_ylabel('LR', color='b')
    axes[1].set_title('LR and val loss together')
    axes[1].legend(loc='upper left');  ax2.legend(loc='upper right')

    plt.tight_layout()
    plt.show()

interact(
    plot_lr_schedule,
    schedule=Dropdown(
        options=[('StepLR', 'step'), ('CosineAnnealingLR', 'cosine'), ('ReduceLROnPlateau', 'plateau')],
        description='Schedule'),
    initial_lr=FloatSlider(value=0.1, min=0.001, max=0.5, step=0.001,
                           description='Initial LR', readout_format='.3f'),
    n_epochs=IntSlider(value=100, min=20, max=200, step=10, description='Epochs'),
    step_size=IntSlider(value=20, min=5, max=50, step=5, description='Step size*'),
    gamma=FloatSlider(value=0.5, min=0.1, max=0.95, step=0.05, description='\u03b3 (decay)'),
    T_max=IntSlider(value=50, min=10, max=100, step=10, description='T_max*'),
    patience=IntSlider(value=10, min=3, max=30, step=1, description='Patience*'),
);

### Think about it

- With CosineAnnealingLR and `T_max=50`, what happens to the learning rate at epoch 50?
  What happens at epoch 100? Is the cycle behaviour desirable?
- ReduceLROnPlateau fires only when val loss stops improving. What is the risk of setting
  `patience` to 2 or 3 epochs on a noisy validation curve (e.g. from a small survey dataset)?
- Which schedule would you choose if you suspected the model might get stuck around epoch 40,
  and you wanted the learning rate to recover and explore again?
- With StepLR and `gamma=0.1`, the lr drops by 10× every `step_size` epochs. After 3 steps,
  the lr is 1000× smaller than the initial value. Is that still useful for training? Why or why not?

---

## Part 4 — Early stopping

Training longer does not always mean a better model.
When the validation loss stops improving and starts rising, the model is overfitting:
it is memorising the training set rather than learning generalisable patterns.

**Early stopping** monitors the validation loss and halts training when
it has not improved for `patience` consecutive epochs.
The model weights from the best validation epoch are saved.

This is not just regularisation — it is also *compute efficiency*:
for a survey telescope's labelled dataset (often small), the overfitting regime
can begin much earlier than the epoch limit suggests.

In [ ]:
# ── Part 4: Early stopping ───────────────────────────────────────────────────

def plot_early_stopping(patience=5, noise_level=0.3, overfit_start=30):
    np.random.seed(42)
    n_epochs = 100
    t = np.arange(n_epochs)

    train_loss = (0.8 * np.exp(-t / 20) + 0.05
                  + 0.02 * np.random.randn(n_epochs))
    val_loss   = (0.8 * np.exp(-t / 20)
                  + 0.15 * np.maximum(0, (t - overfit_start)) / (n_epochs - overfit_start)
                  + 0.05 + noise_level * 0.1 * np.random.randn(n_epochs))
    train_loss = np.clip(train_loss, 0.01, None)
    val_loss   = np.clip(val_loss,   0.01, None)

    best_val, best_epoch, counter, stop_epoch = val_loss[0], 0, 0, None
    for epoch in range(1, n_epochs):
        if val_loss[epoch] < best_val - 1e-6:
            best_val, best_epoch, counter = val_loss[epoch], epoch, 0
        else:
            counter += 1
        if counter >= patience:
            stop_epoch = epoch
            break
    if stop_epoch is None:
        stop_epoch = n_epochs - 1

    show_to = min(stop_epoch + 15, n_epochs - 1)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(train_loss[:show_to + 1], 'b-', lw=2, label='Train loss')
    ax.plot(val_loss[:show_to + 1],   'r-', lw=2, label='Val loss')
    ax.axvline(best_epoch, ls='--', c='green',  lw=1.5, label=f'Best val (epoch {best_epoch})')
    ax.axvline(stop_epoch, ls='--', c='orange', lw=2.0, label=f'Early stop (epoch {stop_epoch})')
    ax.axvspan(best_epoch, stop_epoch, alpha=0.1, color='orange',
               label=f'Patience window ({patience} epochs)')

    ax.set_xlabel('Epoch');  ax.set_ylabel('Loss')
    ax.set_title(f'Early stopping  |  patience={patience}, noise={noise_level:.1f}, overfit start\u2248{overfit_start}')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f'Best val loss {best_val:.4f} at epoch {best_epoch}.')
    print(f'Stopped at epoch {stop_epoch}  ({n_epochs - stop_epoch} epochs unused).')

interact(
    plot_early_stopping,
    patience=IntSlider(value=5, min=1, max=30, step=1, description='Patience'),
    noise_level=FloatSlider(value=0.3, min=0.0, max=2.0, step=0.1, description='Noise'),
    overfit_start=IntSlider(value=30, min=5, max=80, step=5, description='Overfit at'),
);

### Think about it

- Set patience to 20. Does training stop before or after the model has overfit significantly?
  Is there always a clear trade-off between patience and the amount of overfitting tolerated?
- Increase `noise_level` gradually. At what noise level does early stopping start firing
  prematurely — before the validation loss has genuinely flattened?
- A radio survey provides only 500 labelled examples. How does this affect the noise level
  on the validation loss curve, and what setting of `patience` would you recommend?
- Move `overfit_start` to 60. In this case, early stopping saves only a small number of
  epochs — but is the best-epoch model meaningfully better than the final-epoch model?

---

## Exercise — Implement SGD with momentum from scratch

Implement one update step of SGD with momentum in NumPy.
Your implementation will be tested against `torch.optim.SGD(momentum=0.9)` on the
same loss surface used in Part 1.

**Update rule:**

$$v_i \leftarrow \beta \cdot v_i + g_i$$

$$w_i \leftarrow w_i - \alpha \cdot v_i$$

where $g_i$ is the gradient for parameter $i$, $v_i$ is the velocity, $\beta$ is the
momentum coefficient, and $\alpha$ is the learning rate.

In [ ]:
def sgd_momentum_step(params, grads, velocity, lr, momentum):
    """
    Perform one update step of SGD with momentum. Modifies params and velocity in place.

    Args:
        params   : list of np.ndarray -- current parameter arrays
        grads    : list of np.ndarray -- current gradient arrays (same shapes)
        velocity : list of np.ndarray -- running velocity arrays (same shapes)
        lr       : float -- learning rate
        momentum : float -- momentum coefficient (typical value: 0.9)
    Returns:
        None  (params and velocity are updated in place)
    """
    for i in range(len(params)):
        # Step 1: update velocity
        #   velocity[i] = momentum * velocity[i] + grads[i]
        # Step 2: update parameter
        #   params[i] -= lr * velocity[i]
        raise NotImplementedError('Fill in sgd_momentum_step')

In [ ]:
# ── Test: compare your implementation with torch.optim.SGD ──────────────────
np.random.seed(0)
start = np.array([-3.0, 3.0])
lr_test, mom_test, n_test = 0.05, 0.9, 20

# Your implementation
w_yours = start.copy()
v_yours = [np.zeros(2)]
params_yours = [w_yours]

for _ in range(n_test):
    g = grad_f(*w_yours)
    sgd_momentum_step(params_yours, [g], v_yours, lr_test, mom_test)

# PyTorch reference
w_ref = torch.tensor(start, requires_grad=True)
opt_ref = optim.SGD([w_ref], lr=lr_test, momentum=mom_test)

for _ in range(n_test):
    opt_ref.zero_grad()
    loss = w_ref[0]**2 + 10 * w_ref[1]**2
    loss.backward()
    opt_ref.step()

w_ref_np = w_ref.detach().numpy()
print(f'Your result:    w = {w_yours}')
print(f'PyTorch result: w = {w_ref_np}')
print(f'Max difference: {np.max(np.abs(w_yours - w_ref_np)):.2e}')
assert np.allclose(w_yours, w_ref_np, atol=1e-5), \
    'Results do not match. Re-check your velocity update and parameter update.'
print('\u2713 Correct \u2014 your SGD with momentum matches PyTorch.')

---

## Going further

### Part A — Go deeper

Adam's convergence depends critically on $\beta_1$ and $\beta_2$.
The defaults ($\beta_1=0.9$, $\beta_2=0.999$) are well-chosen for most problems,
but there are regimes where different values perform noticeably better or worse.

**Challenge:** Modify `run_adam` in Part 1 to accept $\beta_1$ and $\beta_2$ as arguments
and expose them as sliders in `plot_trajectories`. Explore: (a) what happens as $\beta_1 \to 1.0$
(very high momentum in the first-moment estimate); (b) what happens as $\beta_2 \to 1.0$
(very slow second-moment accumulation). Is there a setting where Adam oscillates worse than SGD?
Can you characterise the failure condition geometrically on the contour plot?

*Suggested approach:* Start with $(\beta_1=0.99, \beta_2=0.999)$ and $(\beta_1=0.9, \beta_2=0.9)$
and compare trajectories on both the elongated bowl (current, factor 10) and a rounder bowl
(reduce the factor toward 1). Notice how the failure mode changes with surface geometry.

### Part B — Lead forward

The optimiser controls *how fast* the network moves through parameter space.
But on a 256×256×3 galaxy image, the *shape* of that parameter space is determined
by the network architecture — specifically, by which computations share parameters.

**Challenge:** Two networks are trained on Galaxy10 with identical Adam settings:
an MLP (flatten → Linear → Linear → Linear) and a CNN (Conv2d blocks).
The CNN converges faster and to a lower validation loss.
Write down two structural reasons — not optimiser reasons — why the CNN's loss landscape
is easier to navigate. Then look at the elongated bowl from Part 1 and sketch what those
structural differences might look like as changes to the contour plot.

*Suggested approach:* Think about how many parameters each architecture uses for the same
task (fewer parameters generally means a lower-dimensional, smoother loss surface),
and what each weight must represent. A weight in a conv filter has a clear local meaning;
a weight in a first Linear layer must somehow encode position as well as pattern.

---

### References

| | |
|---|---|
| Primary | Kingma & Ba (2015) — *Adam: A Method for Stochastic Optimization*. ICLR. [arxiv.org/abs/1412.6980](https://arxiv.org/abs/1412.6980) |
| Blog | Ruder (2016) — *An overview of gradient descent optimisation algorithms*. [ruder.io/optimizing-gradient-descent/](https://ruder.io/optimizing-gradient-descent/) |

---

> **Try the exercise yourself first.** The solution is below — scroll past it if you haven't attempted the exercise.


---

## Solution — `sgd_momentum_step`

SGD with momentum keeps a **running velocity** — a weighted average of all past gradients. At each step, the velocity accumulates the current gradient, then the parameter moves in the direction of the velocity (not the raw gradient).

In [ ]:
def sgd_momentum_step(params, grads, velocity, lr, momentum):
    for i in range(len(params)):
        # Velocity update: exponential moving average of gradients
        # High momentum (0.9) = slow to forget previous direction
        velocity[i] = momentum * velocity[i] + grads[i]

        # Parameter update: step along accumulated velocity
        params[i] -= lr * velocity[i]

# ── What this connects back to ────────────────────────────────────────────
# The convergence curve above shows why momentum helps on the elongated bowl:
# gradients along the shallow axis (w1) are consistently in the same
# direction, so velocity accumulates there → faster convergence along that axis.
# Gradients along the steep axis (w2) oscillate sign → velocity averages out
# the oscillation → less overshoot.
#
# Adam goes further: it normalises the step size per parameter by the square
# root of the squared-gradient running average — Part 2 shows this directly.
